### Chapter 3.2 - Object-Oriented Design for Implementation

Object-oriented design organizes code into objects that carry data and behavior together. This notebook introduces the Python pieces slowly before mapping them to ML training abstractions.


In [ ]:
import math
import random
import numpy as np
import torch


### 3.2.1 Utilities

#### 1. Intuition

A utility is a small helper that supports the main workflow. It is not the model itself.

Object-oriented programming organizes related data and functions into objects. A class is a template. An object is one concrete instance made from that template. An attribute is data stored on an object. A method is a function attached to an object.

`self` is the conventional name for the object receiving a method call.

#### 2. Why this exists

ML projects repeatedly need helpers for saving settings, tracking metrics, plotting, batching, and training. Classes can group these helpers so the code has fewer loose variables.


#### 3. Examples

Plain Python class: store and update a running average.


In [ ]:
class Average:
    def __init__(self):
        self.total = 0.0
        self.count = 0
    def add(self, value):
        self.total += value
        self.count += 1
        return self.total / self.count

meter = Average()
meter.add(3.0)


Use the same object again, so its attributes remember state.


In [ ]:
first = meter.add(5.0)
second = meter.add(7.0)
first, second, meter.total, meter.count


#### 4. Step-by-step breakdown

`class Average:` defines a class template.

`__init__` runs when a new object is created.

`self.total` and `self.count` are attributes stored on the object.

`meter = Average()` creates one object.

`meter.add(3.0)` calls the method. Python passes `meter` as `self` automatically.

The object remembers previous calls because its attributes remain in memory.

#### 5. Connection to ML systems

Training code often uses metric accumulators like this to track average loss or accuracy across batches.

#### 6. Common confusion points

- `self` is not a keyword, but using that name is standard Python style.
- `__init__` initializes object state; it is not called manually in normal use.
- Methods can change attributes, so method calls may have lasting effects.
- Utility classes should make state clearer, not hide important behavior.


### 3.2.2 Models

#### 1. Intuition

A model is a function with parameters learned from data.

In object-oriented code, a model object usually stores parameters as attributes and provides a method for computing predictions.

#### 2. Why this exists

Keeping parameters and prediction logic together reduces mistakes. If weights and bias are separate loose variables, it is easier to pass the wrong one into a function.


#### 3. Examples

Manual model object without PyTorch inheritance.


In [ ]:
class LinearModel:
    def __init__(self):
        self.w = torch.tensor([2.0, -1.0])
        self.b = torch.tensor(0.5)
    def predict(self, X):
        return X @ self.w + self.b

model = LinearModel()
model.predict(torch.tensor([[1.0, 2.0]]))


PyTorch module version. Inheritance means a class reuses behavior from a parent class.


In [ ]:
class TorchLinear(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Linear(2, 1)
    def forward(self, X):
        return self.layer(X)

net = TorchLinear()
net(torch.tensor([[1.0, 2.0]]))


#### 4. Step-by-step breakdown

`LinearModel` is a plain Python class that stores `w` and `b`.

`predict` computes the forward prediction formula.

`class TorchLinear(torch.nn.Module)` means `TorchLinear` inherits from PyTorch's `Module` class.

`super().__init__()` runs the parent class initialization. This lets PyTorch set up internal machinery for parameter tracking.

`forward` defines the computation PyTorch should run when the object is called like `net(X)`.

#### 5. Connection to ML systems

PyTorch models are usually subclasses of `torch.nn.Module`. The parent class manages parameters, nested layers, device movement, and training/evaluation mode.

#### 6. Common confusion points

- Inheritance reuses parent behavior; it is not copying code by hand.
- `super().__init__()` is required so PyTorch can initialize module internals.
- `forward` is called indirectly when you write `net(X)`.
- A model object can contain other layer objects as attributes.


### 3.2.3 Data

#### 1. Intuition

A data object can store features and labels and provide examples when training code asks for them.

An index is a position used to select one item. A dataset often supports `dataset[i]` to return the `i`th example.

#### 2. Why this exists

Training code should not care whether data came from a CSV file, generated tensors, or images on disk. A dataset object gives the training loop a consistent interface.


#### 3. Examples

Plain dataset object with `__len__` and `__getitem__`.


In [ ]:
class TinyData:
    def __init__(self):
        self.X = torch.tensor([[1.0], [2.0], [3.0]])
        self.y = torch.tensor([2.0, 4.0, 6.0])
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

data = TinyData()
data[1]


Create small batches manually from the dataset.


In [ ]:
batch_X = torch.stack([data[0][0], data[1][0]])
batch_y = torch.stack([data[0][1], data[1][1]])
batch_X, batch_y


#### 4. Step-by-step breakdown

`__len__` is a special method Python calls when you use `len(data)`.

`__getitem__` is a special method Python calls when you use square brackets like `data[1]`.

The dataset returns one feature tensor and one label tensor.

`torch.stack` combines several tensors into a new tensor with a batch dimension.

#### 5. Connection to ML systems

PyTorch's `Dataset` and `DataLoader` use these same ideas. A DataLoader repeatedly asks a Dataset for examples and groups them into batches.

#### 6. Common confusion points

- `__getitem__` receives an index, not a feature value.
- A dataset returns examples; a dataloader returns batches.
- `stack` creates a new dimension; it is different from concatenating along an existing dimension.
- Data abstractions should preserve the pairing between each input and its label.


### 3.2.4 Training

#### 1. Intuition

Training means repeatedly changing model parameters to reduce loss.

An epoch is one pass through the training dataset. A batch is the group of examples used in one update. An optimizer is the rule that updates parameters using gradients.

#### 2. Why this exists

The training loop is the control center of supervised learning. It defines what runs first, what repeats, what is stored, and when parameters change.


#### 3. Examples

A minimal training step written directly.


In [ ]:
w = torch.tensor([0.0], requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
X = torch.tensor([[1.0], [2.0]])
y = torch.tensor([2.0, 4.0])
y_hat = X.reshape(-1) * w + b
loss = ((y_hat - y) ** 2).mean()
loss.backward()
w.grad, b.grad


A trainer-like object can group loop settings.


In [ ]:
class TrainerConfig:
    def __init__(self, lr, epochs):
        self.lr = lr
        self.epochs = epochs

config = TrainerConfig(lr=0.03, epochs=3)
config.lr, config.epochs


#### 4. Step-by-step breakdown

The prediction is computed first.

The loss compares predictions with labels.

`loss.backward()` computes gradients for `w` and `b`.

The code stops before updating parameters so the gradient values are visible.

`TrainerConfig` is not a full trainer. It only shows how training settings can be stored in an object.

#### 5. Connection to ML systems

D2L-style `Trainer` objects eventually package the repeated training workflow. Before using them, understand the raw sequence: forward pass, loss, backward pass, parameter update, repeat.

#### 6. Common confusion points

- An epoch is not one gradient update unless the dataset has one batch.
- Gradients are computed before the optimizer update.
- Training objects should not hide concepts you cannot yet explain manually.
- Framework trainers are convenience layers over ordinary control flow.


### 3.2.5 Summary

#### 1. Intuition

Object-oriented design groups related state and behavior.

For ML, common objects include models, datasets, metric trackers, and trainers.

#### 2. Why this exists

The point is not to make code look fancy. The point is to reduce accidental complexity as projects grow.


#### 3. Examples

A tiny object map for a training system.


In [ ]:
objects = {
    "model": "stores parameters and predicts",
    "data": "returns examples or batches",
    "trainer": "runs repeated updates",
}
objects


#### 4. Step-by-step breakdown

Each object has a responsibility.

The model handles prediction.

The data object handles example access.

The trainer handles repetition and updates.

Keeping these responsibilities separate makes later code easier to inspect.

#### 5. Connection to ML systems

Research code frequently defines custom modules, data objects, and training utilities. Reading them is easier when you can identify each object's responsibility.

#### 6. Common confusion points

- Classes are tools for organization, not automatically better code.
- Hidden state can confuse debugging if not named clearly.
- `self` marks object state that persists across method calls.
- Framework abstractions should be mapped back to the manual training loop.


### 3.2.6 Exercises

#### 1. Intuition

These exercises check whether object-oriented code still feels like ordinary Python control flow.

#### 2. Why this exists

If a class confuses you, reduce it to attributes, methods, and method-call order.


#### 3. Examples

Exercise 1: create a metric object and call it twice.


In [ ]:
meter = Average()
a = meter.add(2.0)
b = meter.add(6.0)
a, b


Exercise 2: inspect a PyTorch module's parameters.


In [ ]:
net = TorchLinear()
params = list(net.parameters())
[p.shape for p in params]


#### 4. Step-by-step breakdown

Exercise 1 checks persistent object state.

Exercise 2 checks that PyTorch modules can reveal their trainable parameters.

The object-oriented interface is useful only if you can still trace what data is stored and what computation runs.

#### 5. Connection to ML systems

Later chapters will rely on module and trainer abstractions. These exercises prepare you to read them without treating them as magic.

#### 6. Common confusion points

- Calling a method may change object state.
- `parameters()` comes from PyTorch's parent `Module` class.
- The order of method calls matters in training systems.
- If an abstraction feels unclear, write the manual version beside it.
